In [2]:
import os
os.environ["NO_PROXY"] = "*"

from dotenv import load_dotenv
load_dotenv()

import httpx
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb

api_key = os.environ["GROQ_API_KEY"].strip()
groq_client = Groq(api_key=api_key, http_client=httpx.Client(trust_env=False))
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Клиенты готовы")

C:\Users\PTS\PyCharmMiscProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3174.58it/s]


Клиенты готовы


In [3]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="readme_collection")

data_dir = "data"
all_chunks = []
all_ids = []

for filename in os.listdir(data_dir):
    filepath = os.path.join(data_dir, filename)

    if not os.path.isfile(filepath):  # пропускаем всё, что не файл
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    file_chunks = chunk_text(text)
    for i, chunk in enumerate(file_chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{filename}_chunk_{i}")

embeddings = embed_model.encode(all_chunks)
collection.add(documents=all_chunks, embeddings=embeddings.tolist(), ids=all_ids)

print(f"Загружено {len(all_chunks)} чанков из {len(os.listdir(data_dir))} файлов")

Загружено 32 чанков из 7 файлов


In [6]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="readme_collection")

data_dir = "data"
all_chunks = []
all_ids = []

for filename in os.listdir(data_dir):
    filepath = os.path.join(data_dir, filename)

    if not os.path.isfile(filepath):  # пропускаем всё, что не файл
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    file_chunks = chunk_text(text)
    for i, chunk in enumerate(file_chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{filename}_chunk_{i}")

embeddings = embed_model.encode(all_chunks)
collection.add(documents=all_chunks, embeddings=embeddings.tolist(), ids=all_ids)

print(f"Загружено {len(all_chunks)} чанков из {len(os.listdir(data_dir))} файлов")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte

In [7]:
for filename in os.listdir(data_dir):
    filepath = os.path.join(data_dir, filename)
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            f.read()
        print(f"OK: {filename}")
    except UnicodeDecodeError:
        print(f"ПРОБЛЕМНЫЙ ФАЙЛ: {filename}")

PermissionError: [Errno 13] Permission denied: 'data\\cifar-10-batches-py'

In [8]:
clear

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Получить текущую погоду в указанном городе",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Название города, например 'Москва'"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Какая погода в Frankfurt?"}],
    tools=tools
)

message = response.choices[0].message
print("content:", message.content)
print("tool_calls:", message.tool_calls)

content: None
tool_calls: [ChatCompletionMessageToolCall(id='fc_9a630f49-e041-42e2-babe-12ae14090d1b', function=Function(arguments='{"city":"Frankfurt"}', name='get_weather'), type='function')]


In [5]:
models = groq_client.models.list()
for m in models.data:
    print(m.id)

groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
openai/gpt-oss-20b
qwen/qwen3.6-27b
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
openai/gpt-oss-120b
allam-2-7b
groq/compound


In [9]:
def get_weather(city):
    fake_weather_db = {
    "Москва": "+15C, облачно",
    "Frankfurt": "+18C, солнечно"
    }
    return fake_weather_db.get(city,"Нет данных по этому городу")

import json
tool_call = message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_weather(args["city"])

print("Аргументы",args)
print("Результат функции", result)

messages = [
    {
        "role": "user","content":"Какая погода в Frankfurt?"},
        message,
    {
        "role":"tool",
        "tool_call_id":tool_call.id,
        "content":result
    }

]

final_response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
)

print(final_response.choices[0].message.content)

Аргументы {'city': 'Frankfurt'}
Результат функции +18C, солнечно
Погода в Франкфурте: +18 °C, солнечно.


In [10]:
from ddgs import DDGS

with DDGS() as ddgs:
    results = list(ddgs.text("последние новости про Anthropic Claude", max_results=3))

for r in results:
    print(r["title"])
    print(r["href"])
    print(r.get("body", "")[:200])
    print()


Компания Anthropic заблокировала аккаунты за использование чат-бота ...
https://www.rbc.ru/technology_and_media/12/09/2026/6aa4d5b7d70e7aaa483858a4
Anthropic заблокировала четыре аккаунта пользователей, которые, по версии компании, использовали чат-бот Claude для подготовки контента для российских государственных и финансируемых государством СМИ.

Новости Anthropic: последние анонсы и исследования Claude
https://agenccy.ai/ru/companies/anthropic-news/
Главная / Компании Новости Anthropic Anthropic разрабатывает семейство моделей Claude с акцентом на безопасность ИИ. Этот раздел отслеживает релизы Claude, исследования безопасности, внедрение в бизне

Anthropic обнаружила случаи неправомерного использования Claude
https://news.rambler.ru/tech/57051457-anthropic-obnaruzhila-sluchai-nepravomernogo-ispolzovaniya-claude/
Американская компания Anthropic в четверг опубликовала отчет с перечислением случаев неправомерного использования ИИ-моделей Claude за последние 8 месяцев. В их числе - моше

In [4]:
print(embed_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)


In [5]:
from ddgs import DDGS

def search_readms(question, n_results=3):
    question_embedding = embed_model.encode([question])
    results = collection.query(query_embeddings=question_embedding)
    return "\n\n---\n\n".join(results["documents"][0])


def search_web(query):
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
    return "\n\n".join([f"{r['title']}: {r.get('body', '')}" for r in results])


tools = [
    {
        "type": "function",
        "function": {
            "name": "search_readmes",
            "description": "Искать информацию в README моих собственных GitHub-проектов — используй, если вопрос про технологии, код или содержимое моих репозиториев",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "Вопрос для поиска по README"}
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Искать актуальную информацию в интернете — используй, если вопрос НЕ связан с моими проектами (новости, факты, текущие события)",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Поисковый запрос"}
                },
                "required": ["query"]
            }
        }
    }
]

print("Функции и tools готовы")

Функции и tools готовы


In [6]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Какие технологии используются в моих проектах?"}],
    tools=tools
)

message = response.choices[0].message
print("content:", message.content)
print("tool_calls:", message.tool_calls)

content: None
tool_calls: [ChatCompletionMessageToolCall(id='fc_53c7d888-ac04-40fb-a085-6b5d4afdd89e', function=Function(arguments='{"question":"Какие технологии используются в моих проектах?"}', name='search_readmes'), type='function')]


In [7]:
response2 = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Какая последняя версия Python вышла?"}],
    tools=tools
)

message2 = response2.choices[0].message
print("content:", message2.content)
print("tool_calls:", message2.tool_calls)

content: None
tool_calls: [ChatCompletionMessageToolCall(id='fc_3298f340-0500-4da3-87c8-be375eca5943', function=Function(arguments='{"query":"latest version of Python 2026"}', name='search_web'), type='function')]


In [19]:
import json

def run_agent(user_question, max_steps=5):

    messages = [
        {"role": "system", "content": "У тебя есть доступ к функциям search_readmes и search_web. Используй их для ответа. Не проси пользователя предоставить данные вручную — всегда используй доступные инструменты, чтобы найти информацию самостоятельно."},
        {"role": "user", "content": user_question}]

    for step in range(max_steps):
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=tools  # передаём tools КАЖДЫЙ раз, а не только в первый
        )

        message = response.choices[0].message

        if not message.tool_calls:
            return message.content  # модель готова дать финальный ответ

        tool_call = message.tool_calls[0]
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if function_name == "search_readmes":
            result = search_readms(args["question"])
        elif function_name == "search_web":
            result = search_web(args["query"])
        else:
            result = f"Неизвестная функция: {function_name}"

        messages.append(message)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

    return "Не удалось получить финальный ответ за отведённое число шагов"

print(run_agent("Какая последняя версия PyTorch вышла, и используется ли она в моих проектах?",max_steps=10))

### 1️⃣ Последняя версия PyTorch

| Версия | Дата GA‑релиза | Ключевые новшества |
|--------|----------------|---------------------|
| **PyTorch 2.14.0** | 2 Сентября 2026 г. | • Новый GPU‑backend NVGEMM (авто‑выбор оптимального ядра) <br>• Поддержка FP8/FP4 форматов <br>• Улучшения в `torch.compile` и `torch.fx` <br>• Обновлённый `torchvision` >= 0.19.0 <br>• Улучшенная совместимость с CUDA 12.6 |

Вы можете посмотреть подробные заметки о релизе в официальном блоге:  
<https://pytorch.org/blog/pytorch-2.14-release/>  

---

### 2️⃣ Используется ли PyTorch в ваших проектах?

Я просматривал `README`‑ы всех ваших публичных GitHub‑репозиториев через API `search_readmes`:

| Репозиторий | Указание на PyTorch | Версия, явно указанная в `requirements`/`environment.yml` |
|-------------|---------------------|-----------------------------------------------------------|
| **End-to-End News Classifier (LSTM on PyTorch)** | Да – в описании явно говорится об LSTM‑модели на PyTorch | Не указана кон

In [20]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"].strip()
)

response = llm.invoke("Что такое Docker одним предложением?")
print(response.content)

Docker – это платформа для упаковки, распространения и запуска приложений в изолированных контейнерах.
